# 05 · Data Acquisition — Tanzania Irish potato

Downloads the *Irish Potato Imagery Dataset for Early Detection of Crop Diseases*
(Laizer, Mduma et al., 2023; Zenodo record 8286529, CC BY 4.0): 58,709 smartphone
photos from smallholder farms in Mbeya, Tanzania (Nov 2022 – Apr 2023).

- The three zips (about 7 GB) go to Colab's local disk and are checked against
  Zenodo's checksums.
- File names carry no farm or plant ID, so the capture time is read from each
  photo's EXIF data for every image. It is used to group photos of the same plant
  when the data is split.
- A random sample of `config.TANZANIA_SAMPLE_PER_CLASS` images per class (seed 42)
  is copied to Drive.

In [ ]:
from pathlib import Path
import sys

from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/BilliartMuthoni/confidence-calibrated-crop-diagnosis.git"
REPO_DIR = Path("/content/drive/MyDrive/Sproutly-app")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

ML_DIR = REPO_DIR / "ml"
if str(ML_DIR) not in sys.path:
    sys.path.insert(0, str(ML_DIR))

import config

In [ ]:
# Keep the Drive copy of the repository in sync with GitHub.
!git -C {REPO_DIR} config core.fileMode false
!git -C {REPO_DIR} pull --ff-only
!git -C {REPO_DIR} status --short --branch

## Download the zips to local disk

About 7 GB; expect 10–20 minutes. A zip that is already on local disk and matches
its checksum is not downloaded again.

In [ ]:
from src.data import tanzania

zips = tanzania.download(list(config.TANZANIA_CLASS_MAP), Path("/content/tanzania_raw"))
for name, path in zips.items():
    print(f"{path.name}: {path.stat().st_size / 1e9:.2f} GB, checksum OK")

## Read the capture time of every image

Reads only the start of each photo, where the EXIF data is stored.

In [ ]:
import pandas as pd

frames = []
for name, zip_path in zips.items():
    frame = pd.DataFrame(tanzania.read_inventory(zip_path, name))
    frame.insert(0, "class", config.TANZANIA_CLASS_MAP[name])
    frame.insert(1, "zip", name)
    frames.append(frame)
inventory = pd.concat(frames, ignore_index=True)
inventory["capture_time"] = pd.to_datetime(
    inventory["capture_time"], format="%Y:%m:%d %H:%M:%S", errors="coerce"
)
inventory["capture_day"] = inventory["capture_time"].dt.date

summary = inventory.groupby("class").agg(
    images=("file", "size"),
    with_capture_time=("capture_time", "count"),
    capture_days=("capture_day", "nunique"),
    first=("capture_time", "min"),
    last=("capture_time", "max"),
)
display(summary)
print("Total:", len(inventory))

## Sample and copy to Drive

Starts from an empty folder so a re-run never leaves stale images behind.

In [ ]:
import shutil

if config.TANZANIA_DIR.exists():
    shutil.rmtree(config.TANZANIA_DIR)

sample = (
    inventory.groupby("class", group_keys=False)
    .sample(n=config.TANZANIA_SAMPLE_PER_CLASS, random_state=config.RANDOM_SEED)
    .sort_values(["class", "file"])
    .reset_index(drop=True)
)

for name, zip_path in zips.items():
    class_name = config.TANZANIA_CLASS_MAP[name]
    files = sample.loc[sample["class"] == class_name, "file"].tolist()
    copied = tanzania.extract(zip_path, name, files, config.TANZANIA_DIR / class_name)
    print(f"{class_name}: {copied} of {len(files)} copied")
    assert copied == len(files) == config.TANZANIA_SAMPLE_PER_CLASS

display(sample.groupby("class").agg(images=("file", "size"), capture_days=("capture_day", "nunique")))

## Save the inventories

`tanzania_inventory.csv` lists all 58,709 images with their capture time (for the
data report); `tanzania_sample.csv` lists the sampled images that were copied to Drive.

In [ ]:
inventory.to_csv(config.METADATA_DIR / "tanzania_inventory.csv", index=False)
sample.to_csv(config.METADATA_DIR / "tanzania_sample.csv", index=False)
print("Saved tanzania_inventory.csv and tanzania_sample.csv to", config.METADATA_DIR)

## Save to GitHub

Commits the changed notebook to a new branch and prints a link to open the pull
request (`main` is protected). Change the branch name if this cell has already been
run once. Save the notebook (Ctrl+S) before running this cell.

In [ ]:
from google.colab import userdata

from src import github_sync

print(github_sync.status(REPO_DIR))
print()
print(github_sync.push(
    REPO_DIR,
    branch="colab/tanzania-potato-acquisition",
    message="Acquire a sample of the Tanzania potato field images",
    token=userdata.get("GITHUB_TOKEN"),
    name="BilliartMuthoni",
    email=userdata.get("GIT_EMAIL"),
))